# Лабораторная работа №7: Разработка автономных AI-агентов

## Цель работы
Изучить принципы создания автономных AI-агентов, способных планировать действия, использовать внешние инструменты и выполнять многошаговые задачи.

## Теоретическая часть

### Что такое AI-агент?
AI-агент — это система, которая:
- Воспринимает окружение (входные данные, контекст)
- Планирует последовательность действий для достижения цели
- Использует инструменты (поиск, калькулятор, API и т.д.)
- Выполняет действия и оценивает результаты
- Корректирует план при необходимости

### Архитектура агента
1. **LLM (мозг)** — принимает решения на основе контекста
2. **Планировщик** — разбивает задачу на подзадачи
3. **Инструменты** — функции для выполнения конкретных действий
4. **Память** — хранит историю взаимодействий
5. **Исполнитель** — выполняет выбранные действия

### Популярные фреймворки
- **LangChain/LangGraph** — гибкий фреймворк для создания цепочек и графов агентов
- **AutoGen** (Microsoft) — мульти-агентные системы с диалогом между агентами
- **LlamaIndex** — агенты для работы с документами и данными

## Практическая часть

### Установка зависимостей

In [ ]:
!pip install langchain langchain-openai langgraph python-dotenv -q

### Импорт библиотек и настройка

In [ ]:
import os
from google.colab import userdata

# Для Google Colab: установите API ключ в секреты
# Или используйте локальную переменную окружения
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # Замените на ваш ключ

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

### Создание инструментов для агента

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Вычисляет математическое выражение."""
    try:
        result = eval(expression)
        return f"Результат: {result}"
    except Exception as e:
        return f"Ошибка: {str(e)}"

@tool
def weather_checker(city: str) -> str:
    """Проверяет погоду в городе (симуляция)."""
    # В реальном проекте здесь был бы вызов API погоды
    weather_data = {
        "москва": "+15°C, облачно",
        "санкт-петербург": "+12°C, дождь",
        "новосибирск": "+8°C, ясно",
        "екатеринбург": "+10°C, ветрено"
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return f"Погода в {city}: {weather_data[city_lower]}"
    else:
        return f"Погода в {city}: данные недоступны (симуляция)"

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    """Конвертирует валюту (симуляция курсов)."""
    rates = {
        "USD": 1.0,
        "EUR": 0.85,
        "RUB": 90.0,
        "GBP": 0.73
    }
    if from_currency.upper() not in rates or to_currency.upper() not in rates:
        return "Неизвестная валюта"
    
    amount_in_usd = amount / rates[from_currency.upper()]
    result = amount_in_usd * rates[to_currency.upper()]
    return f"{amount} {from_currency} = {result:.2f} {to_currency}"

tools = [calculator, weather_checker, currency_converter]

### Создание и запуск агента

In [ ]:
# Инициализация LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Создание агента с использованием ReAct паттерна
agent = create_react_agent(llm, tools)

# Функция для запуска агента
def run_agent(query: str):
    messages = [HumanMessage(content=query)]
    response = agent.invoke({"messages": messages})
    return response["messages"][-1].content

# Примеры запросов
queries = [
    "Какая погода в Москве?",
    "Сколько будет 25 * 4 + 100?",
    "Конвертируй 1000 рублей в доллары США",
    "Какая погода в Новосибирске и сколько будет 15 + 27?"
]

In [ ]:
# Тестирование агента
for query in queries:
    print(f"\n{'='*50}")
    print(f"Запрос: {query}")
    print(f"{'='*50}")
    response = run_agent(query)
    print(f"Ответ агента: {response}")

### Задание 1: Расширение функционала агента

Добавьте новые инструменты:
1. Поиск информации в интернете (используйте DuckDuckGo или другой бесплатный API)
2. Работа с датами (калькулятор дат, определение дня недели)
3. Генерация случайных чисел с заданными параметрами

In [ ]:
# Ваш код здесь
# Добавьте новые инструменты и протестируйте их работу

### Задание 2: Создание мульти-агентной системы

Создайте систему из нескольких специализированных агентов:
- **Аналитик** — анализирует данные и делает выводы
- **Калькулятор** — выполняет вычисления
- **Исследователь** — ищет информацию

Реализуйте взаимодействие между агентами для решения сложных задач.

In [ ]:
# Ваш код здесь
# Создайте мульти-агентную систему

### Задание 3: Работа с памятью агента

Реализуйте сохранение истории диалога и использование контекста из предыдущих взаимодействий.

In [ ]:
from langchain.memory import ConversationBufferMemory

# Ваш код здесь
# Реализуйте память для агента

## Контрольные вопросы

1. Какие основные компоненты входят в архитектуру AI-агента?
2. В чем разница между одноагентными и мульти-агентными системами?
3. Какие паттерны планирования вы знаете (ReAct, Plan-and-Execute и др.)?
4. Как агент выбирает, какой инструмент использовать?
5. Какие проблемы могут возникнуть при создании агентов и как их решать?

## Требования к отчету

1. Описание реализованных инструментов и их назначения
2. Примеры работы агента с различными запросами
3. Анализ успешности выполнения заданий (какие задачи решены, какие нет)
4. Ответы на контрольные вопросы
5. Выводы о возможностях и ограничениях современных AI-агентов